<a href="https://colab.research.google.com/github/BilalKhaliqWillis/BILAL-Assignment2/blob/main/P2_BILAL_Final_Project_Natural_Language_Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 2: Chatbot Prediction

In [ ]:
!pip install -q nltk tensorflow keras numpy

In [1]:
import random
import json
import pickle
import numpy as np
import nltk

# DOWNLOADS
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from tensorflow.keras.models import load_model

lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [3]:
from google.colab import files
uploaded = files.upload()

Saving chatbot_model.keras to chatbot_model.keras
Saving classes.pkl to classes.pkl
Saving medical_intents.json to medical_intents (1).json
Saving words.pkl to words.pkl


In [4]:
intents = json.load(open('medical_intents.json'))
words = pickle.load(open('words.pkl', 'rb'))
classes = pickle.load(open('classes.pkl', 'rb'))
model = load_model('chatbot_model.keras')

In [5]:
def clean_up_sentence(sentence):
    sentence_words = word_tokenize(sentence)
    sentence_words = [lemmatizer.lemmatize(word.lower()) for word in sentence_words]
    return sentence_words

In [6]:
def get_bow_of_words(sentence, words):
    sentence_words = clean_up_sentence(sentence)
    bag = [0] * len(words)

    for s in sentence_words:
        for i, w in enumerate(words):
            if w == s:
                bag[i] = 1

    return np.array(bag)

In [7]:
def predict_class(sentence):
    bow = get_bow_of_words(sentence, words)
    res = model.predict(np.array([bow]), verbose=0)[0]

    ERROR_THRESHOLD = 0.25

    results = [[i, r] for i, r in enumerate(res) if r > ERROR_THRESHOLD]
    results.sort(key=lambda x: x[1], reverse=True)

    return [{"intent": classes[r[0]], "probability": str(r[1])} for r in results]

In [8]:
def get_model_response(intents_list, intents_json):
    if len(intents_list) == 0:
        return "Sorry, I did not understand that."

    tag = intents_list[0]['intent']

    for intent in intents_json['intents']:
        if intent['tag'] == tag:
            return random.choice(intent['responses'])

In [9]:
def chatbot_response(msg):
    intents_list = predict_class(msg)
    return get_model_response(intents_list, intents)

In [10]:
# Testing Responses
print(chatbot_response("Hi"))
print(chatbot_response("I am suffering from cold"))
print(chatbot_response("I took sudafed tablet"))
print(chatbot_response("I am still not feeling well"))
print(chatbot_response("I want to consult a doctor"))
print(chatbot_response("Can you book it for me"))
print(chatbot_response("Patient Name: McDonal Date: 19900918 Doctor: Ayoola Hospital: The Ottawa Hospital"))
print(chatbot_response("Thank you for the help"))

Hello!, How can I help?
You should take a sudafed tablet after your meal.
Good, now take some rest and then let me know if you are feeling better after having sudafed tablet.
Ok, its not just a normal cold, you should consult a doctor.
Book an appointment.
yes please state time and doctor's name in the following format; 'Patient Name:  Date:  Doctor: Hospital:
appointment details saved. you shall be contacted shortly.
Welcome
